# Radiomics + Clinical Survival Analysis

This notebook predicts **time-to-event** using three feature sets, compared side by side:
- **Radiomics only**
- **Clinical only**
- **Radiomics + Clinical combined**

For each feature set, four survival models are trained and evaluated:
- **Random Survival Forest (RSF)**
- **Gradient Boosting Survival Analysis (GBSA)**
- **Survival Support Vector Machine (Survival SVM)**
- **Penalized Cox Model** (elastic-net regularized Cox regression, `CoxnetSurvivalAnalysis`)

Evaluation uses **stratified 5-fold cross-validation on a training set** plus a single
**held-out test set** (same patients held out for every feature set, for a fair
comparison), with the **concordance index (C-index)** and **log-rank test** as metrics.
A summary CSV comparing all feature-set/model combinations is saved at the end.

**Expected inputs**

1. `radiomics_features.csv` — one row per scan (first PET scan of each patient):
   `image_id, feature_1, feature_2, ..., feature_n`

2. `clinical_events.csv` — one row per patient (time-to-event labels):
   `image_id, patient_id, time, event`
   - `time`: time to event or censoring (ideally > 0; non-positive values are auto-corrected)
   - `event`: 1 = event occurred, 0 = censored

3. `clinical_data.csv` — one row per patient (clinical variables used as predictors), e.g.:
   `patient_id, age, sex, stage, smoking_status, ...`
   - Can contain a mix of numeric (age, lab values) and categorical (sex, stage) columns —
     both are handled automatically (see Section 5).

Edit the file paths and column names in the **Config** cell below, then run all cells.


## 1. Setup — install & import packages

In [ ]:
# Uncomment to install (run once)
# %pip install scikit-survival lifelines scikit-learn pandas numpy matplotlib seaborn --quiet


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.svm import FastSurvivalSVM
from sksurv.linear_model import CoxnetSurvivalAnalysis

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
N_SPLITS = 5          # requested cross-validation folds (auto-reduced if too few events)
HOLDOUT_SIZE = 0.2    # fraction reserved as final untouched test set


## 2. Config — set your file paths and column names

In [ ]:
# ---- EDIT THESE ----
RADIOMICS_CSV      = "radiomics_features.csv"   # image_id, feature_1, feature_2, ...
CLINICAL_EVENT_CSV = "clinical_events.csv"      # image_id, patient_id, time, event
CLINICAL_DATA_CSV  = "clinical_data.csv"        # patient_id, age, sex, stage, ...

ID_COL          = "image_id"     # radiomics / event join key
PATIENT_ID_COL  = "patient_id"   # event / clinical-data join key
TIME_COL        = "time"
EVENT_COL       = "event"

# If your clinical data file is keyed by image_id instead of patient_id, set this to ID_COL
CLINICAL_JOIN_COL = PATIENT_ID_COL

RESULTS_CSV = "survival_model_results.csv"   # output summary file

# If None, the epsilon used to replace time <= 0 is chosen automatically as
# half of the smallest strictly-positive observed time. Set a fixed number
# (in the same units as your `time` column) to override.
TIME_EPSILON = None


## 3. Load & merge data

We keep only the **first PET scan per patient**, merge radiomics features with the
time/event table on `image_id`, then merge in the clinical variables on
`CLINICAL_JOIN_COL` (patient-level, by default).


In [ ]:
radiomics_df = pd.read_csv(RADIOMICS_CSV)
events_df    = pd.read_csv(CLINICAL_EVENT_CSV)
clinical_df  = pd.read_csv(CLINICAL_DATA_CSV)

print("Radiomics shape:", radiomics_df.shape)
print("Events shape   :", events_df.shape)
print("Clinical shape :", clinical_df.shape)

radiomics_df.head()


In [ ]:
events_df.head()


In [ ]:
clinical_df.head()


In [ ]:
# Keep the feature-name lists from each *raw* file before merging, so we always know
# which merged columns are radiomics vs. clinical, regardless of overlapping names.
radiomics_feature_cols = [c for c in radiomics_df.columns if c != ID_COL]
clinical_feature_cols_raw = [c for c in clinical_df.columns if c != CLINICAL_JOIN_COL]

print(f"Radiomics features: {len(radiomics_feature_cols)}")
print(f"Clinical features : {len(clinical_feature_cols_raw)} -> {clinical_feature_cols_raw}")


In [ ]:
# If a patient has multiple scans, keep the first PET scan only
if PATIENT_ID_COL in events_df.columns:
    events_df = (events_df
                 .sort_values(ID_COL)
                 .drop_duplicates(subset=PATIENT_ID_COL, keep="first"))

# 1) radiomics + events (on image_id)
df = pd.merge(events_df, radiomics_df, on=ID_COL, how="inner")

# 2) + clinical data (on patient_id, or image_id if CLINICAL_JOIN_COL == ID_COL)
df = pd.merge(df, clinical_df, on=CLINICAL_JOIN_COL, how="left")

print("Merged shape:", df.shape)
n_missing_clinical = df[clinical_feature_cols_raw].isna().all(axis=1).sum()
if n_missing_clinical > 0:
    print(f"Warning: {n_missing_clinical} row(s) have no matching clinical data (all clinical columns NaN) "
          f"-- check that {CLINICAL_JOIN_COL} values match between files.")

assert TIME_COL in df.columns, f"{TIME_COL} not found after merge"
assert EVENT_COL in df.columns, f"{EVENT_COL} not found after merge"
df[[ID_COL, PATIENT_ID_COL, TIME_COL, EVENT_COL]].head()


## 4. Clean survival time

`scikit-survival` requires **strictly positive** observed times, so `time <= 0` cannot be
passed through as-is. Rather than dropping those rows, we **replace non-positive times
with a small positive epsilon** (auto-derived as half the smallest positive observed
time, or fixed via `TIME_EPSILON`). Rows with entirely missing `time`/`event` are still
dropped since there's nothing sensible to substitute.


In [ ]:
before = len(df)

df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce")
df[EVENT_COL] = pd.to_numeric(df[EVENT_COL], errors="coerce")

missing_mask = df[TIME_COL].isna() | df[EVENT_COL].isna()
n_missing = missing_mask.sum()
if n_missing > 0:
    print(f"Dropping {n_missing} row(s) with missing time or event:")
    print(df.loc[missing_mask, [ID_COL, TIME_COL, EVENT_COL]])
df = df.loc[~missing_mask].reset_index(drop=True)

positive_times = df.loc[df[TIME_COL] > 0, TIME_COL]
if TIME_EPSILON is None:
    epsilon = positive_times.min() / 2 if not positive_times.empty else 1e-3
else:
    epsilon = TIME_EPSILON

nonpositive_mask = df[TIME_COL] <= 0
n_adjusted = nonpositive_mask.sum()
if n_adjusted > 0:
    print(f"\nReplacing {n_adjusted} row(s) with time <= 0 by epsilon = {epsilon:g}:")
    print(df.loc[nonpositive_mask, [ID_COL, TIME_COL, EVENT_COL]])
    df.loc[nonpositive_mask, TIME_COL] = epsilon

print(f"\nRows before: {before}, after cleaning: {len(df)} (dropped {n_missing}, adjusted {n_adjusted})")


## 5. Preprocessing

We build **three separate feature matrices** that all share the same row order/index:

- `X_radiomics` — radiomics features only (numeric, imputed + standardized).
- `X_clinical` — clinical features only. Numeric columns are imputed (median) and
  standardized; categorical columns are imputed (most frequent) and one-hot encoded.
- `X_combined` — radiomics + processed clinical features, concatenated.

Keeping the row order identical across all three lets us reuse the exact same
train/holdout split and cross-validation folds for a fair comparison.


In [ ]:
time  = df[TIME_COL].astype(float).values
event = df[EVENT_COL].astype(bool).values
print(f"Events: {event.sum()} / {len(event)} ({event.mean()*100:.1f}%)")

y = Surv.from_arrays(event=event, time=time)


In [ ]:
# --- Radiomics preprocessing ---
X_radiomics_raw = df[radiomics_feature_cols].apply(pd.to_numeric, errors="coerce")
constant_radiomics_cols = X_radiomics_raw.columns[X_radiomics_raw.nunique(dropna=True) <= 1]
X_radiomics_raw = X_radiomics_raw.drop(columns=constant_radiomics_cols)
print(f"Dropped {len(constant_radiomics_cols)} constant/non-numeric radiomics columns")

radiomics_imputer = SimpleImputer(strategy="median")
radiomics_scaler = StandardScaler()
X_radiomics = pd.DataFrame(
    radiomics_scaler.fit_transform(radiomics_imputer.fit_transform(X_radiomics_raw)),
    columns=X_radiomics_raw.columns,
    index=df.index
)
print(f"X_radiomics shape: {X_radiomics.shape}")
X_radiomics.head()


In [ ]:
# --- Clinical preprocessing ---
clinical_raw = df[clinical_feature_cols_raw].copy()

numeric_clinical_cols = clinical_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_clinical_cols = [c for c in clinical_feature_cols_raw if c not in numeric_clinical_cols]

print(f"Numeric clinical columns    : {numeric_clinical_cols}")
print(f"Categorical clinical columns: {categorical_clinical_cols}")

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

clinical_preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_clinical_cols),
    ("cat", categorical_transformer, categorical_clinical_cols),
])

clinical_processed = clinical_preprocessor.fit_transform(clinical_raw)

clinical_feature_names = list(numeric_clinical_cols)
if categorical_clinical_cols:
    ohe = clinical_preprocessor.named_transformers_["cat"].named_steps["onehot"]
    clinical_feature_names += list(ohe.get_feature_names_out(categorical_clinical_cols))

X_clinical = pd.DataFrame(clinical_processed, columns=clinical_feature_names, index=df.index)
print(f"X_clinical shape: {X_clinical.shape}")
X_clinical.head()


In [ ]:
# --- Combined feature set ---
X_combined = pd.concat([X_radiomics, X_clinical], axis=1)
print(f"X_combined shape: {X_combined.shape}")

feature_sets = {
    "Radiomics only": X_radiomics,
    "Clinical only": X_clinical,
    "Radiomics + Clinical": X_combined,
}


## 6. Kaplan-Meier estimator

Overall survival curve for the full cohort.

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(durations=time, event_observed=event, label="Overall cohort")

fig, ax = plt.subplots(figsize=(7, 5))
kmf.plot_survival_function(ax=ax, ci_show=True)
ax.set_xlabel("Time")
ax.set_ylabel("Survival probability")
ax.set_title("Kaplan-Meier Survival Curve — Overall Cohort")
plt.tight_layout()
plt.show()

print(f"Median survival time: {kmf.median_survival_time_}")


### Kaplan-Meier stratified by a clinical variable (optional)

If a categorical clinical column exists (e.g. sex, stage), this plots survival curves
by group with a log-rank test — a natural first check of whether that variable matters
before feeding it into a model.


In [ ]:
if categorical_clinical_cols:
    example_clinical_cat = categorical_clinical_cols[0]
    groups = df[example_clinical_cat].fillna("Missing")

    fig, ax = plt.subplots(figsize=(7, 5))
    for group_val in groups.unique():
        mask = (groups == group_val).values
        if mask.sum() == 0:
            continue
        kmf_g = KaplanMeierFitter()
        kmf_g.fit(time[mask], event[mask], label=str(group_val))
        kmf_g.plot_survival_function(ax=ax)

    ax.set_xlabel("Time")
    ax.set_ylabel("Survival probability")
    ax.set_title(f"Kaplan-Meier by {example_clinical_cat}")
    plt.tight_layout()
    plt.show()
else:
    print("No categorical clinical columns found -- skipping this example plot.")


### Kaplan-Meier stratified by a radiomics feature (optional)

Median split on the first radiomics feature, with a log-rank test.


In [ ]:
example_feature = radiomics_feature_cols[0] if radiomics_feature_cols else None

if example_feature is not None and example_feature in df.columns:
    median_val = df[example_feature].median()
    high_group = df[example_feature] >= median_val
    low_group  = ~high_group

    fig, ax = plt.subplots(figsize=(7, 5))

    kmf_high = KaplanMeierFitter()
    kmf_high.fit(time[high_group], event[high_group], label=f"High {example_feature}")
    kmf_high.plot_survival_function(ax=ax)

    kmf_low = KaplanMeierFitter()
    kmf_low.fit(time[low_group], event[low_group], label=f"Low {example_feature}")
    kmf_low.plot_survival_function(ax=ax)

    ax.set_xlabel("Time")
    ax.set_ylabel("Survival probability")
    ax.set_title(f"Kaplan-Meier by {example_feature} (median split)")
    plt.tight_layout()
    plt.show()

    result = logrank_test(time[high_group], time[low_group],
                           event[high_group], event[low_group])
    print(f"Log-rank test p-value: {result.p_value:.4f}")


## 7. Holdout split + stratified cross-validation setup

We split **once**, on row indices, so the same patients end up in the train/CV set and
the holdout set for *every* feature set — this keeps the comparison fair. Folds are
stratified on event status so every fold contains both events and censored cases.


In [ ]:
idx_all = np.arange(len(df))
idx_trainval, idx_holdout = train_test_split(
    idx_all, test_size=HOLDOUT_SIZE, random_state=RANDOM_STATE, stratify=event
)

event_trainval = event[idx_trainval]
event_holdout = event[idx_holdout]
time_trainval = time[idx_trainval]
time_holdout = time[idx_holdout]
y_trainval = y[idx_trainval]
y_holdout = y[idx_holdout]

print(f"Train/CV set: {len(idx_trainval)} patients ({event_trainval.sum()} events)")
print(f"Holdout set : {len(idx_holdout)} patients ({event_holdout.sum()} events)")

n_events_trainval = int(event_trainval.sum())
n_censored_trainval = len(event_trainval) - n_events_trainval
safe_splits = max(2, min(N_SPLITS, n_events_trainval, n_censored_trainval))
if safe_splits < N_SPLITS:
    print(f"\nWarning: only {n_events_trainval} events / {n_censored_trainval} censored "
          f"cases in the train/CV set; reducing N_SPLITS from {N_SPLITS} to {safe_splits} "
          f"so every fold contains at least one event.")
    N_SPLITS = safe_splits
if n_events_trainval < 2 or n_censored_trainval < 2:
    raise ValueError(
        "Too few events or censored cases to run cross-validation reliably. "
        "Consider gathering more data or using leave-one-out validation instead."
    )

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
# Precompute fold indices once so every feature set is evaluated on identical folds
cv_fold_indices = list(kf.split(np.zeros(len(event_trainval)), event_trainval))
print(f"Using {N_SPLITS} folds (shared across all feature sets)")


## 8. Model builder

- **Random Survival Forest**, **Gradient Boosting Survival**, **Survival SVM** as before.
- **Penalized Cox (Elastic Net)** — `CoxnetSurvivalAnalysis`; its regularization strength
  is picked via a stratified inner grid search, refit separately for each feature set
  (since the useful alpha range depends on the data/dimensionality).


In [ ]:
def build_inner_cv(X_for_cv, y_for_cv, n_splits=3):
    """Stratified inner CV splits (by event status) for the Coxnet alpha search."""
    n_events = int(y_for_cv["event"].sum())
    n_censored = len(y_for_cv) - n_events
    safe_n = max(2, min(n_splits, n_events, n_censored))
    skf_inner = StratifiedKFold(n_splits=safe_n, shuffle=True, random_state=RANDOM_STATE)
    return list(skf_inner.split(X_for_cv, y_for_cv["event"]))


def make_models(X_for_cv, y_for_cv, n_alphas=15):
    """Build a fresh dict of models, with a Coxnet alpha grid tailored to X_for_cv."""
    try:
        pathfinder = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01, max_iter=100_000)
        pathfinder.fit(X_for_cv, y_for_cv)
        full_alpha_path = pathfinder.alphas_
    except Exception:
        full_alpha_path = np.logspace(-3, 1, n_alphas)

    step = max(1, len(full_alpha_path) // n_alphas)
    alpha_grid = full_alpha_path[::step]
    inner_cv = build_inner_cv(X_for_cv, y_for_cv, n_splits=3)

    return {
        "Random Survival Forest": RandomSurvivalForest(
            n_estimators=300, min_samples_split=10, min_samples_leaf=15,
            max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE
        ),
        "Gradient Boosting Survival": GradientBoostingSurvivalAnalysis(
            n_estimators=200, learning_rate=0.05, max_depth=3,
            subsample=0.8, random_state=RANDOM_STATE
        ),
        "Survival SVM": FastSurvivalSVM(
            alpha=1.0, rank_ratio=1.0, max_iter=1000, tol=1e-6, random_state=RANDOM_STATE
        ),
        "Penalized Cox (Elastic Net)": GridSearchCV(
            CoxnetSurvivalAnalysis(l1_ratio=0.9, max_iter=100_000),
            param_grid={"alphas": [[a] for a in alpha_grid]},
            cv=inner_cv, error_score=0.5, n_jobs=1,
        ),
    }


## 9. Pipeline runner

Runs cross-validation + final holdout evaluation for one feature set, using the fold
indices precomputed in Section 7 so every feature set sees the exact same folds/patients.


In [ ]:
def run_feature_set(X, feature_set_name):
    X_trainval = X.iloc[idx_trainval].reset_index(drop=True)
    X_holdout_fs = X.iloc[idx_holdout].reset_index(drop=True)

    cv_records = []
    for fold_i, (train_idx, val_idx) in enumerate(cv_fold_indices, start=1):
        X_tr, X_val = X_trainval.iloc[train_idx], X_trainval.iloc[val_idx]
        y_tr, y_val = y_trainval[train_idx], y_trainval[val_idx]

        models = make_models(X_tr, y_tr)
        for name, model in models.items():
            model.fit(X_tr, y_tr)
            c_index = model.score(X_val, y_val)
            cv_records.append({"feature_set": feature_set_name, "model": name,
                                "fold": fold_i, "c_index": c_index})
        print(f"[{feature_set_name}] fold {fold_i}/{len(cv_fold_indices)} done")

    cv_df_fs = pd.DataFrame(cv_records)

    final_models = make_models(X_trainval, y_trainval)
    holdout_records = []
    risk_scores = {}

    for name, model in final_models.items():
        model.fit(X_trainval, y_trainval)
        c_index_holdout = model.score(X_holdout_fs, y_holdout)
        rs = model.predict(X_holdout_fs)
        risk_scores[name] = rs

        median_risk = np.median(rs)
        high_risk = rs >= median_risk
        low_risk = ~high_risk
        if high_risk.sum() > 0 and low_risk.sum() > 0:
            lr = logrank_test(time_holdout[high_risk], time_holdout[low_risk],
                               event_holdout[high_risk], event_holdout[low_risk])
            p_value = lr.p_value
        else:
            p_value = np.nan

        holdout_records.append({"feature_set": feature_set_name, "model": name,
                                 "holdout_c_index": c_index_holdout, "holdout_logrank_p": p_value})
        print(f"[{feature_set_name}] {name:28s} | Holdout C-index = {c_index_holdout:.3f} | log-rank p = {p_value:.4f}")

    holdout_df_fs = pd.DataFrame(holdout_records)
    return cv_df_fs, holdout_df_fs, final_models, risk_scores, X_holdout_fs


## 10. Run all three feature sets

This is the main, most time-consuming cell: it trains all 4 models, with nested CV for
the penalized Cox model, for each of the 3 feature sets (12 model fits x folds total).


In [ ]:
all_cv = []
all_holdout = []
all_final_models = {}
all_risk_scores = {}
all_X_holdout = {}

for fs_name, X_fs in feature_sets.items():
    print(f"\n=== Feature set: {fs_name} ({X_fs.shape[1]} features) ===")
    cv_df_fs, holdout_df_fs, final_models_fs, risk_scores_fs, X_holdout_fs = run_feature_set(X_fs, fs_name)
    all_cv.append(cv_df_fs)
    all_holdout.append(holdout_df_fs)
    all_final_models[fs_name] = final_models_fs
    all_risk_scores[fs_name] = risk_scores_fs
    all_X_holdout[fs_name] = X_holdout_fs

cv_df_master = pd.concat(all_cv, ignore_index=True)
holdout_df_master = pd.concat(all_holdout, ignore_index=True)


## 11. Compare feature sets and models

In [ ]:
cv_summary_master = (cv_df_master
                     .groupby(["feature_set", "model"])["c_index"]
                     .agg(["mean", "std"])
                     .reset_index()
                     .rename(columns={"mean": "cv_c_index_mean", "std": "cv_c_index_std"}))

results_summary = pd.merge(cv_summary_master, holdout_df_master, on=["feature_set", "model"])
results_summary = results_summary[[
    "feature_set", "model", "cv_c_index_mean", "cv_c_index_std", "holdout_c_index", "holdout_logrank_p"
]].sort_values(["model", "holdout_c_index"], ascending=[True, False]).reset_index(drop=True)

results_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=results_summary, x="model", y="holdout_c_index", hue="feature_set", ax=ax)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Random (0.5)")
ax.set_ylabel("Holdout C-index")
ax.set_xlabel("")
ax.set_title("Holdout C-index by Model and Feature Set")
ax.set_ylim(0.4, 1.0)
plt.xticks(rotation=15, ha="right")
plt.legend(title="Feature set", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=cv_df_master, x="model", y="c_index", hue="feature_set", ax=ax)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("C-index (CV validation folds)")
ax.set_xlabel("")
ax.set_title(f"{N_SPLITS}-Fold Cross-Validation — C-index by Model and Feature Set")
plt.xticks(rotation=15, ha="right")
plt.legend(title="Feature set", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 12. Feature importance — Radiomics + Clinical (combined set)

Bars are colored by modality (radiomics vs. clinical) so you can see at a glance which
type of data the model is relying on most.


In [ ]:
top_n = 20
combined_final_models = all_final_models["Radiomics + Clinical"]
X_holdout_combined = all_X_holdout["Radiomics + Clinical"]

def modality_of(feature_name):
    return "Clinical" if feature_name in X_clinical.columns else "Radiomics"

rsf_combined = combined_final_models["Random Survival Forest"]
perm_result = permutation_importance(
    rsf_combined, X_holdout_combined, y_holdout, n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1
)
rsf_importance = pd.DataFrame({
    "feature": X_holdout_combined.columns,
    "importance": perm_result.importances_mean
})
rsf_importance["modality"] = rsf_importance["feature"].apply(modality_of)
rsf_importance = rsf_importance.sort_values("importance", ascending=False).head(top_n)

fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.3)))
colors = rsf_importance["modality"].map({"Radiomics": "#4C72B0", "Clinical": "#DD8452"})
ax.barh(rsf_importance["feature"][::-1], rsf_importance["importance"][::-1], color=colors[::-1])
ax.set_xlabel("Drop in C-index (permutation importance)")
ax.set_title(f"RSF Feature Importance — Combined Model (Top {top_n})")

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#4C72B0", "#DD8452"]]
ax.legend(handles, ["Radiomics", "Clinical"], loc="lower right")
plt.tight_layout()
plt.show()


## 13. Penalized Cox coefficients — Radiomics + Clinical (combined set)

Signed coefficients from the elastic-net Cox model, colored by modality. Positive =
associated with higher risk; negative = associated with lower risk (protective).


In [ ]:
cox_combined = combined_final_models["Penalized Cox (Elastic Net)"].best_estimator_
best_alpha = combined_final_models["Penalized Cox (Elastic Net)"].best_params_["alphas"][0]
print(f"Selected alpha: {best_alpha:.5f}")

cox_coefs = pd.DataFrame({
    "feature": X_holdout_combined.columns,
    "coefficient": cox_combined.coef_.ravel()
})
cox_coefs["modality"] = cox_coefs["feature"].apply(modality_of)
n_nonzero = (cox_coefs["coefficient"] != 0).sum()
print(f"Non-zero coefficients: {n_nonzero} / {len(cox_coefs)}")

cox_coefs_nonzero = (cox_coefs[cox_coefs["coefficient"] != 0]
                     .assign(abs_coef=lambda d: d["coefficient"].abs())
                     .sort_values("abs_coef", ascending=False)
                     .head(top_n))

if not cox_coefs_nonzero.empty:
    fig, ax = plt.subplots(figsize=(8, max(4, len(cox_coefs_nonzero) * 0.3)))
    colors = ["#d62728" if c > 0 else "#1f77b4" for c in cox_coefs_nonzero["coefficient"][::-1]]
    bars = ax.barh(cox_coefs_nonzero["feature"][::-1], cox_coefs_nonzero["coefficient"][::-1], color=colors)
    # annotate modality next to each bar label
    labels = [f"{feat} ({mod})" for feat, mod in
              zip(cox_coefs_nonzero["feature"][::-1], cox_coefs_nonzero["modality"][::-1])]
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Coefficient (log hazard ratio, standardized features)")
    ax.set_title(f"Penalized Cox — Top {len(cox_coefs_nonzero)} Non-zero Coefficients (Combined)")
    plt.tight_layout()
    plt.show()
else:
    print("All coefficients were shrunk to zero at the selected alpha -- try a smaller alpha_min_ratio or lower l1_ratio.")


## 14. Risk-group Kaplan-Meier curves on holdout (feature set x model)

In [ ]:
fs_names = list(feature_sets.keys())
model_names_list = list(next(iter(all_final_models.values())).keys())

fig, axes = plt.subplots(len(fs_names), len(model_names_list),
                          figsize=(5 * len(model_names_list), 4.5 * len(fs_names)),
                          sharey=True)

for row, fs_name in enumerate(fs_names):
    for col, model_name in enumerate(model_names_list):
        ax = axes[row, col]
        risk_scores = all_risk_scores[fs_name][model_name]
        median_risk = np.median(risk_scores)
        high_risk = risk_scores >= median_risk
        low_risk = ~high_risk

        kmf_hr = KaplanMeierFitter()
        kmf_hr.fit(time_holdout[high_risk], event_holdout[high_risk], label="High risk")
        kmf_hr.plot_survival_function(ax=ax, ci_show=False)

        kmf_lr = KaplanMeierFitter()
        kmf_lr.fit(time_holdout[low_risk], event_holdout[low_risk], label="Low risk")
        kmf_lr.plot_survival_function(ax=ax, ci_show=False)

        p_val = holdout_df_master.loc[
            (holdout_df_master["feature_set"] == fs_name) & (holdout_df_master["model"] == model_name),
            "holdout_logrank_p"
        ].values[0]

        ax.set_title(f"{fs_name}\n{model_name}\np={p_val:.3f}", fontsize=9)
        ax.legend(fontsize=7)
        if row == len(fs_names) - 1:
            ax.set_xlabel("Time")
        if col == 0:
            ax.set_ylabel("Survival probability")

plt.tight_layout()
plt.show()


## 15. Save summary results to CSV

In [ ]:
results_summary.to_csv(RESULTS_CSV, index=False)
print(f"Saved results to {RESULTS_CSV}")
results_summary


## Notes

- **Fair comparison across feature sets**: Section 7 splits patients into train/CV vs.
  holdout **once**, and Section 10 reuses those exact same row indices and the same
  stratified CV fold assignments for all three feature sets — so differences in
  C-index reflect the *features*, not a lucky/unlucky split.
- **Clinical preprocessing** (Section 5): numeric clinical columns are median-imputed
  and standardized; categorical columns (e.g. sex, tumor stage) are imputed with the
  most frequent category and one-hot encoded. This happens automatically based on each
  column's dtype — check the printed `numeric_clinical_cols` / `categorical_clinical_cols`
  to confirm they were detected correctly (a numeric-looking code like a stage stored as
  `1`/`2`/`3` will be treated as numeric; rename/cast it first if you want it treated as
  categorical instead).
- **"All samples are censored" fix**: both outer and inner CV are stratified by event
  status, and `N_SPLITS` auto-reduces if there aren't enough events for the requested
  fold count.
- **Zero/negative time handling**: non-positive times are replaced with a small epsilon
  rather than dropped (Section 4).
- **C-index**: 0.5 = random ranking, 1.0 = perfect ranking of predicted risk vs. actual
  event order. **Log-rank p-value**: whether the model's high- vs low-risk holdout
  groups have statistically distinguishable survival curves (p < 0.05 = yes).
- If **Clinical only** outperforms **Radiomics only**, or vice versa, that's informative
  on its own; if **Combined** doesn't beat the better of the two individually, the two
  modalities may be capturing overlapping/redundant information, or the combined feature
  space may be too high-dimensional for the current sample size (consider stronger
  regularization -- e.g. lower `alpha_min_ratio` / higher `l1_ratio` for Coxnet -- or
  feature selection before combining).
